# Prepare Dataset Splits

This notebook converts Pascal VOC annotations into an internal JSON format
used by the training and evaluation notebooks.

Goals:
- parse Pascal VOC annotations
- convert annotations into an internal dataset structure
- split the dataset into train / validation / test sets
- save the splits as JSON files

In [ ]:
from collections import Counter
import json
import random

import matplotlib.pyplot as plt
import numpy as np

from project_config import (
    PROJECT_ROOT,
    IMAGES_DIR,
    ANNOTATIONS_DIR,
    SPLITS_DIR,
    SEED,
)
from utils.voc import parse_voc_xml

In [ ]:
print("PROJECT_ROOT:", PROJECT_ROOT)
print("IMAGES_DIR exists:", IMAGES_DIR.exists())
print("ANNOTATIONS_DIR exists:", ANNOTATIONS_DIR.exists())

xml_paths = sorted(ANNOTATIONS_DIR.glob("*.xml"))

print("XML files:", len(xml_paths))
print("Sample XML:", xml_paths[0] if xml_paths else "none")

if not xml_paths:
    raise ValueError(f"No XML annotation files found in: {ANNOTATIONS_DIR}")

## 1. Class mapping and internal dataset representation

This step performs:

1. class name to integer ID mapping
2. loading all XML annotations
3. checking whether the corresponding image exists
4. filtering unknown classes
5. creating the internal dataset representation

Each dataset item contains:

- image path
- filename
- image dimensions
- bounding boxes
- class names
- class IDs
- number of objects

In [ ]:
CLASS_NAME_TO_ID = {
    "1": 0,
    "2": 1,
    "3": 2,
    "4": 3,
    "5": 4,
    "6": 5,
}

ID_TO_CLASS_NAME = {v: k for k, v in CLASS_NAME_TO_ID.items()}
CLASS_NAME_TO_ID

In [ ]:
dataset = []
missing_images = []
unknown_classes = Counter()

for xml_path in xml_paths:
    ann = parse_voc_xml(xml_path)
    image_path = IMAGES_DIR / ann["filename"]

    if not image_path.exists():
        missing_images.append(str(image_path))
        continue

    boxes = []
    class_names = []
    class_ids = []

    for obj in ann["objects"]:
        class_name = obj["class_name"]
        bbox = obj["bbox"]

        if class_name not in CLASS_NAME_TO_ID:
            unknown_classes[class_name] += 1
            continue

        boxes.append(bbox)
        class_names.append(class_name)
        class_ids.append(CLASS_NAME_TO_ID[class_name])

    if len(boxes) == 0:
        continue

    dataset.append({
        "image_path": str(image_path),
        "filename": ann["filename"],
        "width": ann["width"],
        "height": ann["height"],
        "boxes": boxes,
        "class_names": class_names,
        "class_ids": class_ids,
        "num_objects": len(boxes),
    })

print("Valid samples:", len(dataset))
print("Missing images:", len(missing_images))
print("Unknown classes:", dict(unknown_classes))

if not dataset:
    raise ValueError("Dataset is empty after filtering.")

dataset[0]

In [ ]:
sorted_by_objects = sorted(dataset, key=lambda x: x["num_objects"], reverse=True)

for item in sorted_by_objects[:10]:
    print(item["filename"], "| objects:", item["num_objects"])

## 2. Split the dataset into train / validation / test

The dataset is divided into three subsets:

- **train** – used for model training
- **validation** – used for hyperparameter tuning
- **test** – used only for final evaluation

This notebook uses the following split ratio:

- **70%** training
- **15%** validation
- **15%** test

The dataset is shuffled before splitting to avoid bias caused by file order.
A fixed random seed is used to ensure reproducibility.

In [ ]:
random.seed(SEED)

dataset_shuffled = dataset.copy()
random.shuffle(dataset_shuffled)

n = len(dataset_shuffled)
n_train = int(0.70 * n)
n_val = int(0.15 * n)
n_test = n - n_train - n_val

train_data = dataset_shuffled[:n_train]
val_data = dataset_shuffled[n_train:n_train + n_val]
test_data = dataset_shuffled[n_train + n_val:]

print("Train:", len(train_data))
print("Val:", len(val_data))
print("Test:", len(test_data))

In [ ]:
def count_classes(split):
    counter = Counter()
    for item in split:
        for cls in item["class_names"]:
            counter[cls] += 1
    return counter

train_counter = count_classes(train_data)
val_counter = count_classes(val_data)
test_counter = count_classes(test_data)

print("Train:", train_counter)
print("Val:", val_counter)
print("Test:", test_counter)

In [ ]:
classes = ["1", "2", "3", "4", "5", "6"]
train_counts = [train_counter.get(c, 0) for c in classes]
val_counts = [val_counter.get(c, 0) for c in classes]
test_counts = [test_counter.get(c, 0) for c in classes]

x = np.arange(len(classes))
width = 0.25

plt.figure(figsize=(10, 5))
plt.bar(x - width, train_counts, width, label="train")
plt.bar(x, val_counts, width, label="val")
plt.bar(x + width, test_counts, width, label="test")

plt.xticks(x, classes)
plt.xlabel("Dice value")
plt.ylabel("Count")
plt.title("Class distribution across splits")
plt.legend()
plt.grid(axis="y")
plt.show()

## 3. Save dataset splits

Each split is saved as a JSON file containing the dataset records
in the same internal format.

The following files are created in `data/splits/`:

- `train.json`
- `val.json`
- `test.json`

In [ ]:
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

with open(SPLITS_DIR / "train.json", "w", encoding="utf-8") as f:
    json.dump(train_data, f, indent=2)

with open(SPLITS_DIR / "val.json", "w", encoding="utf-8") as f:
    json.dump(val_data, f, indent=2)

with open(SPLITS_DIR / "test.json", "w", encoding="utf-8") as f:
    json.dump(test_data, f, indent=2)

print("Saved splits to:", SPLITS_DIR)